In [ ]:
# For pallas kernel
!pip install "jax[tpu]==0.4.35" -f https://storage.googleapis.com/jax-releases/libtpu_releases.html --force-reinstall

Looking in links: https://storage.googleapis.com/jax-releases/libtpu_releases.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.3/62.3 kB 6.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 77.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 128.9/128.9 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.3/87.3 MB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 137.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 94.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 47.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.9/71.9 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.3/133.3 kB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.

In [ ]:
import functools
from functools import partial
from typing import Callable

import jax
from jax.experimental import pallas as pl
import jax.experimental.pallas.mosaic_gpu as plgpu
from jax.experimental.pallas import tpu as pltpu
from jax import random
import jax.numpy as jnp
from jax import numpy as jnp
import numpy as np

from scaling_transformer.data import (
    create_test_dataset,
    create_training_dataset,
    datapoint_to_string,
    tokenizer,
)
from scaling_transformer.evaluation import eval_model
from scaling_transformer.experiments import (
    estimate_params_and_flops,
    get_config_for_model_size,
    run_experiment,
    transformer_experiment,
)
from scaling_transformer.fused_moe import (
    fused_matmul,
    fused_matmul_kernel,
    get_expert_metadata,
    get_raw_expert_and_block,
    update_dead_threads,
    vmapped_expert_metadata,
)
from scaling_transformer.training import (
    adam_opt_update,
    adam_update,
    cross_entropy_loss,
    get_batch,
    get_decayed_lr,
    init_adam_state,
    init_train_state,
    loss_fn,
    train,
    train_step,
)
from scaling_transformer.transformer import (
    apply_attention,
    apply_block,
    apply_embedding,
    apply_layer_norm,
    apply_linear,
    apply_model,
    init_params,
    init_transformer_block,
)

print(jax.devices())


[TpuDevice(id=0, process_index=0, coords=(0,0,0), core_on_chip=0)]


1. Dataset creation functions



In [ ]:
from scaling_transformer.data import create_test_dataset, create_training_dataset, datapoint_to_string, tokenizer


Functions to initialize transformer

In [ ]:
from scaling_transformer.transformer import init_params, init_transformer_block


Pallas Kernels

In [ ]:
from scaling_transformer.fused_moe import fused_matmul, fused_matmul_kernel, get_expert_metadata, get_raw_expert_and_block, update_dead_threads, vmapped_expert_metadata


The forward functions

In [ ]:
from scaling_transformer.transformer import apply_attention, apply_block, apply_embedding, apply_layer_norm, apply_linear, apply_model


Training loop functions

In [ ]:
from scaling_transformer.training import adam_opt_update, adam_update, cross_entropy_loss, get_batch, get_decayed_lr, init_adam_state, init_train_state, loss_fn, train, train_step


The evaluation functions

In [ ]:
from scaling_transformer.evaluation import eval_model


In [ ]:
from scaling_transformer.experiments import run_experiment


Experiment dashboard

In [ ]:
import numpy as np

test_set = create_training_dataset(50000).tolist()
test_set = [datapoint_to_string(example) for example in test_set]
test_set = [tokenizer(example) for example in test_set]
test_set = jnp.array(np.array(test_set))

In [ ]:
from scaling_transformer.experiments import estimate_params_and_flops, get_config_for_model_size, transformer_experiment


In [ ]:
import gc

jax.clear_caches()
gc.collect()

naive_config = {
  'num_tokens': 5000,
  'epochs': 1,
  'batch_size': 32,
  'max_lr': 1.0e-5,
  'decay_alpha': 0.1,
  'norm_eps': 1e-5,
  'mini_test_set': None,
  'vocab_size': 13,
  'hidden_dim': 512,
  'mlp_dim': 2048,
  'attention_dim': 32,
  'num_qheads': 4,
  'num_kvheads': 2,
  'num_layers': 1,
  'num_experts': 128,
  'top_k': 1,
  'load_balancing_alpha': 0.01,
  'use_custom_kernel': False,
  'modal_gpu': 'A100',
  'block_sizes': {
      'b': 256,
      'd': 512,
      'f': 256,
  },
  'param_seed': 1010101
}

fused_config = {
  'num_tokens': 5000,
  'epochs': 1,
  'batch_size': 32,
  'max_lr': 1.0e-5,
  'decay_alpha': 0.1,
  'norm_eps': 1e-5,
  'mini_test_set': None,
  'vocab_size': 13,
  'hidden_dim': 512,
  'mlp_dim': 2048,
  'attention_dim': 32,
  'num_qheads': 4,
  'num_kvheads': 2,
  'num_layers': 1,
  'num_experts': 128,
  'top_k': 1,
  'load_balancing_alpha': 0.01,
  'use_custom_kernel': True,
  'modal_gpu': 'A100',
  'block_sizes': {
      'b': 256,
      'd': 512,
      'f': 256,
  },
  'param_seed': 1010101
}

params, _ = run_experiment(naive_config, jit=True)


At step 0/156, average loss was 4.1342644691467285


In [ ]:
import numpy as np
experiment_batch_size = 32768

test_set = create_training_dataset(experiment_batch_size).tolist()
test_set = [datapoint_to_string(example) for example in test_set]
test_set = [tokenizer(example) for example in test_set]
test_set = jnp.array(np.array(test_set))

naive_config['batch_size'] = experiment_batch_size
fused_config['batch_size'] = experiment_batch_size

naive_forward = jax.jit(partial(apply_model, batch=test_set, config=naive_config))
fused_forward = jax.jit(partial(apply_model, batch=test_set, config=fused_config))

x = jax.block_until_ready(naive_forward(params))
y = jax.block_until_ready(fused_forward(params))

%timeit -n 1 -r 100 jax.block_until_ready(naive_forward(params))
%timeit -n 1 -r 100 jax.block_until_ready(fused_forward(params))

1.6 s ± 2.61 ms per loop (mean ± std. dev. of 100 runs, 1 loop each)
105 ms ± 156 µs per loop (mean ± std. dev. of 100 runs, 1 loop each)


In [ ]:
jnp.allclose(x[0],y[0], atol=1e-2, rtol=1e-20)

Array(True, dtype=bool)